In [1]:
# -*- coding: utf-8 -*-
"""
Created on Tue Dec 19 11:46:51 2023

@author: Thoma
"""

import pandas
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import time


In [2]:

# Chemin vers le fichier chromedriver (pour Chrome)
driver_path = 'C:/Users/Thoma/App../data/Local/Programs/Spyder/Python/chromedriver'

# Options du navigateur Chrome
chrome_options = webdriver.ChromeOptions()

#chrome_options.add_argument('headless')

# Créer une instance du navigateur
driver = webdriver.Chrome(options=chrome_options)

# Ouvrir la page webhttps://fr.kompass.com/a/services-aux-entreprises/80/v/clermont-ferrand/fr_83_63_63113/
#driver.get("https://www.helloasso.com/e/recherche?tab=associations&bbox=-7.316687829134622%2C42.10613515649942%2C10.657959706435463%2C51.21356451029922&category_tags=sport&category_tags=loisirs")
#driver.get("https://www.helloasso.com/e/recherche?tab=associations&bbox=-6.4760793971694%2C42.1061351564999%2C9.817351274467%2C51.21356451029985&category_tags=sport")
driver.get("https://www.helloasso.com/e/recherche/associations?page=1&category_tags=sport&place_city=Toulouse&place_department=Haute-Garonne")

# Mettre en plein écran la fenêtre du navigateur
driver.maximize_window()
driver.implicitly_wait(12)



def is_element_on_page(driver, xpath):
    try:
        # Recherchez l'élément par XPath
        element = driver.find_element(By.XPATH, xpath)
        
        # Si l'élément est trouvé, renvoie True
        return element is not None
    except:
        # Si une exception est levée (élément non trouvé), renvoie False
        return False




In [ ]:


data_list = []
k=1
i=0
t=0

while t < 16 : 
    passer = driver.find_element(By.XPATH, '//*[@id="LayoutDefault"]/main/div/div[3]/div[2]/div/button[2]')
    driver.implicitly_wait(10)
    time.sleep(3)
    driver.execute_script("arguments[0].click();", passer)
    t+=1
time.sleep(0.5)
    
while  i<50:
    print(i)
    k=1
    while (k<31) :

        try :
            # Attendez que la page soit complètement chargée (vous pouvez ajuster le délai selon les besoins)
            print(k)
            # Simuler le clic sur l'image
            lien_asso = driver.find_element(By.XPATH, f'//*[@id="results"]/ul/li[{k}]/div/a/div/div[1]/p[1]')

            nom_association = lien_asso.text.strip()
            nom_association = nom_association.replace(' ', '-')
            nom_association = nom_association.lower()
            nom_association = nom_association.replace('é', 'e')
            nom_association = nom_association.replace('è', 'e')
            nom_association = nom_association.replace('à', 'a')
            nom_association = nom_association.replace("'", '-')
            print(nom_association)
            
            # Ouvrir le lien dans une nouvelle fenêtre
            driver.execute_script(f"window.open('https://www.helloasso.com/associations/{nom_association}', '_blank');")
            
            new_window_handle = driver.window_handles[-1]
            driver.switch_to.window(new_window_handle)
            # Attendre un peu avant de poursuivre
            time.sleep(0.5)
            if  (is_element_on_page(driver, '//*[@id="carousel-prod_organizations"]/div[1]/div[1]/div/h2') ) :
                # Fermez le navigat/eur
                driver.back()
                # Fermer la nouvelle fenêtre
                driver.close()
                
                # Une fois que vous avez terminé avec la nouvelle fenêtre, vous pouvez basculer vers la fenêtre principale
                driver.switch_to.window(driver.window_handles[0])
                
                # Basculer vers la fenêtre principale
                k+=1

            else :
                # Simuler le clic sur le bouton "Afficher l'adresse"
                wait = WebDriverWait(driver, 1)
                afficher_adresse_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="contact"]/div[2]/div[2]/div[1]/div[2]/div/div/button')))
                driver.execute_script("arguments[0].click();", afficher_adresse_button)
                # Attendez que la page soit complètement chargée après le deuxième clic
                # Maintenant, vous pouvez extraire les informations de la page, y compris les adresses mail
                # Simuler le clic sur un lien avec "email" dans l'attribut href
                email_link = driver.find_element(By.XPATH, '//*[@id="contact"]/div[2]/div[2]/div/div[2]/div/div/p')
                email_address = email_link.text
                print(email_address)
                # Imprimer l'adresse mail
                #email_link.click()	

                data_list.append({'Adresse mail': email_address, 'Nom association': nom_association})
                # Fermez le navigat/eur
                driver.back()
                # Fermer la nouvelle fenêtre
                driver.close()
                
                # Une fois que vous avez terminé avec la nouvelle fenêtre, vous pouvez basculer vers la fenêtre principale
                driver.switch_to.window(driver.window_handles[0])
                
                # Basculer vers la fenêtre principale
                k+=1
           
        except NoSuchElementException:
            driver.back()
            # Fermer la nouvelle fenêtre
            driver.close()
            
            # Une fois que vous avez terminé avec la nouvelle fenêtre, vous pouvez basculer vers la fenêtre principale
            driver.switch_to.window(driver.window_handles[0])
            
            # Basculer vers la fenêtre principale
            k+=1
            
        except TimeoutException:
            driver.back()
            # Fermer la nouvelle fenêtre
            driver.close()
            
            # Une fois que vous avez terminé avec la nouvelle fenêtre, vous pouvez basculer vers la fenêtre principale
            driver.switch_to.window(driver.window_handles[0])
            
            # Basculer vers la fenêtre principale
            k+=1
            
    # Créer undataFrame à partir dedata_list
    df = pandas.DataFrame(data_list)

    try:
        df_base_de_donnees = pandas.read_excel('../data/scraping/Flyte/base_de_donnees.xlsx')
        # Supprimer les adresses en commun de df 
        df = df[~df['Adresse mail'].isin(df_base_de_donnees['Adresse mail'])]

    except FileNotFoundError:
        print("Le fichier base_de_donnees.xlsx n'a pas été trouvé")
        df_base_de_donnees = pandas.dataFrame()

    try:
        # Charger ledataFrame existant
        df_mails_disponibles = pandas.read_excel('../data/scraping/Flyte/mails_disponibles.xlsx')
        
        # Supprimer les adresses en commun
        df = df[~df['Adresse mail'].isin(df_mails_disponibles['Adresse mail'])]

    except FileNotFoundError:
        print("Le fichier mails_disponibles.xlsx n'a pas été trouvé")
        df_mails_disponibles = pandas.dataFrame()

    # Supprimer les doublons basés sur la colonne 'Adresse mail'
    df_sans_doublons = df.drop_duplicates(subset=['Adresse mail'], keep='first')

    try:
        # Enregistrer ledataFrame mis à jour dans un nouveau fichier Excel
        df_sans_doublons.to_excel('../data/scraping/Flyte/resultats_code.xlsx', index=False)
    except Exception as e:
        print(f"Erreur lors de l'enregistrement du fichier: {e}")

    print('quoi')
    passer = driver.find_element(By.XPATH, '//*[@id="LayoutDefault"]/main/div/div[3]/div[2]/div/button[2]')
    driver.implicitly_wait(10)
    time.sleep(0.2)
    driver.execute_script("arguments[0].click();", passer)
    i+=1




0
1
asc-cnes---section-basket
2
jinenkan-ninjutsu-toulouse
3
comite-regional-occitanie-de-la-federation-française-de-danse
4
tolosa-bureau-des-sports
5
tfcc
6
balma-velo-sprint
7
jst-pradettes
8
agepac
9
tuc-roller-sports
10
tocsad
11
linkrope
12
club-sportif-de-l-association-valentin-haüy-de-haute-garonne
13
as-insa-toulouse
14
nektar-krew
15
toulouse-universite-club
16
apsr
17
as-actiphypsy
18
club-des-supporters-du-stade-toulousain-"le-huit"
19
ut1rugby-capitole
20
association-française-de-functional-fitness
21
ecole-long-tao
22
escapade-club
23
france-freestyle-ball
24
les-gros-braqueurs
25
toulouse-haltero-club
26
immersion-libre
27
tuc-tennis
28
union-des-jeunes-sportifs
29
100%-latino-dance-factory
30
bds-ict
quoi
1
1
petanque-7-deniers
2
toulouse-club-patinage
3
club-alpin-de-toulouse
4
bal-au-dessus-des-3000
5
delta-ceps
6
raspaille-et-cochonnaille
7
volley-sourd-toactuc
8
cvifs
9
vivre-airsoft
10
ctvmovement
11
wolves-toulouse-cheerleading
12
le-collectif-des-sportives
13
str

In [ ]:
# Créer undataFrame à partir dedata_list
dff = pandas.DataFrame(data_list)

# Charger ledataFrame existant pour la base de données
df_base_de_donnees = pandas.read_excel('../data/scraping/Flyte/base_de_donnees.xlsx')

# Supprimer les adresses en commun de df
df = df[~df['Adresse mail'].isin(df_base_de_donnees['Adresse mail'])]

# Charger ledataFrame existant pour les mails disponibles
df_mails_disponibles = pandas.read_excel('../data/scraping/Flyte/mails_disponibles.xlsx')

# Supprimer les adresses en commun de df
df = df[~df['Adresse mail'].isin(df_mails_disponibles['Adresse mail'])]

# Supprimer les doublons basés sur la colonne 'Adresse mail'
df_sans_doublons = df.drop_duplicates(subset=['Adresse mail'], keep='first')

# Enregistrer ledataFrame mis à jour dans un nouveau fichier Excel
df_sans_doublons.to_excel('../data/scraping/Flyte/resultats_code.xlsx', index=False)